# Introduction to Groupby

## Notebook Outline
* [Introduction to Groupby](#IntroToGroupby)
* [Calculating Simple Statistics on Groups](#simplegroupstats)
* [Calculating Statistics on Multiple Columns in Groups, using .agg()](#agg)

# Google Colab Path Note

If you run this notebook in Google Colab, mount Google Drive first:

```python
from google.colab import drive
drive.mount('/content/drive')
BASE_DIR = '/content/drive/MyDrive/Colab Notebooks/UCI/UCI_427.62_Python_for_Data_Analysis'
DATA_DIR = f'{BASE_DIR}/05_Datasets'
ASSIGNMENTS_DIR = f'{BASE_DIR}/04_Assignments'
OUTPUTS_DIR = f'{BASE_DIR}/04_Assignments/Outputs'
```

If you run locally in Jupyter/Anaconda, do not use `/content/drive`; use relative paths such as `../05_Datasets/...`.
The Mac path `/Users/yuzhang/.../My Drive/...` is only for local file browsing and is not a Colab path.


In [1]:
import pandas as pd
import numpy as np

# Introduction to Groupby

The groupby method allows to group data in a dataframe by one or more columns to create a 'groupby object'. We can then analyze each of the groups to create per-group-statistics and analysis.  For example, the mean temperature per month (this involves grouping by the month) or the total sales per shift manager (this involves grouping by the manager). Let's look at some examples!

<https://pandas.pydata.org/pandas-docs/stable/groupby.html>

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
filepath = ('/content/drive/MyDrive/Colab Notebooks/UCI/UCI_427.62_Python_for_Data_Analysis/05_Datasets/M1_724080-13739-2001')
headers = ['Year', 'Month', 'Day', 'Hour', 'Air Temp', 'Dew Point Temp', 'Sea Level Pressure',
           'Wind Direction', 'Wind Speed Rate',
           'Sky Condition Total Coverage Code',
           'Liquid Precipitation Depth Dimension - 1Hr Duration',
           'Liquid Precipitation Depth Dimension - Six Hour Duration']
weatherData = pd.read_csv(filepath, delim_whitespace=True,
                          names=headers)
weatherData.loc[:, 'Air Temp'] = (weatherData['Air Temp']/10) * 1.8 + 32
weatherData.head(2)

/tmp/ipykernel_532/775506768.py:7: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  weatherData = pd.read_csv(filepath, delim_whitespace=True,
/tmp/ipykernel_532/775506768.py:9: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[46.4 47.3 48.2 49.1 50.  50.9 51.8 52.7 53.6 54.5 55.4 56.3 57.2 58.1
 59.  59.9 60.8 61.7 62.6 63.5 64.4 65.3 66.2 67.1 46.4 47.3 48.2 49.1
 50.  50.9 51.8 52.7 53.6 54.5 55.4 56.3 57.2 58.1 59.  59.9 60.8 61.7
 62.6 63.5 64.4 65.3 66.2 67.1 46.4 47.3 48.2 49.1 50.  50.9 51.8 52.7
 53.6 54.5 55.4 56.3 57.2 58.1 59.  59.9 60.8 61.7 62.6 63.5 64.4 65.3
 66.2 67.1]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  weatherData.loc[:, 'Air Temp'] = (weatherData['Air Temp']/10) * 1.8 + 32


,Year,Month,Day,Hour,Air Temp,Dew Point Temp,Sea Level Pressure,Wind Direction,Wind Speed Rate,Sky Condition Total Coverage Code,Liquid Precipitation Depth Dimension - 1Hr Duration,Liquid Precipitation Depth Dimension - Six Hour Duration
0,2022,1,1,0,46.4,80,10120,180,35,1,0,0
1,2022,1,1,1,47.3,80,10120,180,35,1,0,0


#### Use .groupby() to group the data by the month

In [4]:
districts_new2 = weatherData.groupby(['Year','Month','Day']).agg(['mean'])

In [5]:
df_new=  weatherData.groupby(['Year','Month','Day']).apply(lambda weatherData: (weatherData['Air Temp']))

/tmp/ipykernel_532/3270605807.py:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_new=  weatherData.groupby(['Year','Month','Day']).apply(lambda weatherData: (weatherData['Air Temp']))


In [6]:
df = weatherData[['Day']]*2

In [7]:
print(df)

    Day
0     2
1     2
2     2
3     2
4     2
..  ...
67    6
68    6
69    6
70    6
71    6

[72 rows x 1 columns]


In [8]:
print(df_new)

Year  Month  Day    
2022  1      1    0     46.4
                  1     47.3
                  2     48.2
                  3     49.1
                  4     50.0
                        ... 
             3    67    63.5
                  68    64.4
                  69    65.3
                  70    66.2
                  71    67.1
Name: Air Temp, Length: 72, dtype: float64


#### Let's explore our groupby object using the .groups attribute and the .size() method.
The .groups attribute will list the name of each group and the row index values that make up each group.

The .size() method will print the size of each group

In [12]:
weatherData['Date_Time_Combined'] = pd.to_datetime(weatherData[['Year', 'Month', 'Day', 'Hour']])
monthGroups = weatherData.groupby(by=weatherData['Date_Time_Combined'].dt.month)
monthGroups.groups

{1: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71]}

In [13]:
# it's not surprising that the February group has a smaller size since there
# are less days in February
monthGroups.size()

,0
Date_Time_Combined,
1,72


In [14]:
monthGroups['Air Temp'].describe()

,count,mean,std,min,25%,50%,75%,max
Date_Time_Combined,,,,,,,,
1,72.0,56.75,6.273688,46.4,51.575,56.75,61.925,67.1


#### Getting groups using the .get_group() method
We can get a specific group by using the .get_group method. We just need to pass the name of the group which will be one of the values that we grouped by. See the example below

In [15]:
# To get the March group we do the following:
marchData = monthGroups.get_group(3)
marchData.head(3)

KeyError: 3

In [16]:
monthGroups.groups.keys()

dict_keys([1])

# Calculating Simple Statistics on Groups

#### Now let's use the .mean(), .min(), and .max() methods on the groupby object to find the mean, min, and max air temperature of each month.

Note that operating on a groupby will return a DataFrame or Series depending on what the operation is.

In [17]:
monthlyMeanTemps = monthGroups['Air Temp'].mean()
print(type(monthlyMeanTemps))
monthlyMeanTemps

<class 'pandas.core.series.Series'>


,Air Temp
Date_Time_Combined,
1,56.75


In [18]:
monthGroups['Air Temp'].max()

,Air Temp
Date_Time_Combined,
1,67.1


In [19]:
monthGroups['Air Temp'].min()

,Air Temp
Date_Time_Combined,
1,46.4


#### We found a missing value! As a review, let's use some of what we have learned to take a closer look and fix it!

First let's use .loc[] and boolean series to find all values less than 10 (which is already 6 degrees lower than the minimum temp in January). The reason we do this is that I want to see if there are any other bad data points.

In [20]:
weatherData.loc[weatherData['Air Temp'] < 10, :]

,Year,Month,Day,Hour,Air Temp,Dew Point Temp,Sea Level Pressure,Wind Direction,Wind Speed Rate,Sky Condition Total Coverage Code,Liquid Precipitation Depth Dimension - 1Hr Duration,Liquid Precipitation Depth Dimension - Six Hour Duration,Date_Time_Combined


#### Let's replace it with the a linear interpolation between hour 7 and hour 9..
There are a few ways we can do this. For now, I am just going to use the .loc method to grab the rows corresponding to 2001-2-26 hours 7 and 9.

As a review, I will do this in a few steps:
* First I will use .loc to get the appropriate rows.
* Then, I will use .loc to get the rows for only the 'Air Temp' column
* Then, I will use the .mean() to get the mean value (same as a basic linear interpolation) all in one line.

In [21]:
weatherData.loc[(weatherData['Month'] == 2) & (weatherData['Day'] == 26) &
                (weatherData['Hour'].isin([7, 9])), :]

,Year,Month,Day,Hour,Air Temp,Dew Point Temp,Sea Level Pressure,Wind Direction,Wind Speed Rate,Sky Condition Total Coverage Code,Liquid Precipitation Depth Dimension - 1Hr Duration,Liquid Precipitation Depth Dimension - Six Hour Duration,Date_Time_Combined


In [22]:
weatherData.loc[(weatherData['Month'] == 2) & (weatherData['Day'] == 26) &
                (weatherData['Hour'].isin([7, 9])), 'Air Temp']

,Air Temp


In [23]:
weatherData.loc[(weatherData['Month'] == 2) & (weatherData['Day'] == 26) &
                (weatherData['Hour'].isin([7, 9])), 'Air Temp'].mean()

nan

#### Let's fill the bad value with this estimated value, using the .loc method.
Note that the row label of the row with the bad value is 1352. We will use that. Remember this is not the only way, this is just one way.

Note that the '\' allows me to continue the line of code on the next line. I will explain in lecture.

Also note that you must spell your columns names correctly - or you may accidentally add a new column!

In [24]:
weatherData.loc[1352, 'Air Temp'] = \
    weatherData.loc[(weatherData['Month'] == 2) & (weatherData['Day'] == 26) &
                    (weatherData['Hour'].isin([7, 9])), 'Air Temp'].mean()

The `AssertionError` likely arises from `NaT` values introduced in the `Date_Time_Combined` column when `weatherData` was expanded to index `1352`. These `NaT`s can cause inconsistencies during `groupby` operations. Let's drop rows where `Date_Time_Combined` is `NaT` to clean the DataFrame before re-creating the `monthGroups` object.

In [26]:
# Drop rows where 'Date_Time_Combined' is NaT
weatherData.dropna(subset=['Date_Time_Combined'], inplace=True)

# Recreate monthGroups after cleaning weatherData
monthGroups = weatherData.groupby(by=weatherData['Date_Time_Combined'].dt.month)

# Now, re-run the min() aggregation
monthGroups['Air Temp'].min()

,Air Temp
Date_Time_Combined,
1,46.4


#### Finally, find the minimum temps of each month.

In [27]:
monthGroups['Air Temp'].min()

,Air Temp
Date_Time_Combined,
1,46.4


# Using the .agg() method on a groupby objects
We can use the .agg() (short for aggregate) method on a group by object to calculate multiple statistics. Let's use the monthGroups object we have already created.

Docs are here: <https://pandas.pydata.org/pandas-docs/stable/generated/pandas.core.groupby.DataFrameGroupBy.agg.html>

In [28]:
monthGroups['Air Temp'].agg(['min', 'mean', 'max', 'std'])

,min,mean,max,std
Date_Time_Combined,,,,
1,46.4,56.75,67.1,6.273688


#### Other strings that pandas will recognize as functions are here:
<http://pandas.pydata.org/pandas-docs/stable/basics.html> (Scroll down about 25% to find the screenshot below)
![](functionnames.png)

#### Using .agg to compute different statistics on different columns:

In [29]:
monthGroups.agg({'Air Temp': 'mean', 'Wind Speed Rate': 'max'})

,Air Temp,Wind Speed Rate
Date_Time_Combined,,
1,56.75,35.0


#### Using .agg to compute different statistics on different columns, and renaming the columns:

In [30]:
(monthGroups.agg({'Air Temp': 'mean', 'Wind Speed Rate': 'max'}).
 rename(columns={'Air Temp': 'Mean Air Temp', 'Wind Speed Rate': 'Max Wind Speed Rate'}))

,Mean Air Temp,Max Wind Speed Rate
Date_Time_Combined,,
1,56.75,35.0


# Using the apply method for custom analysis
The apply method allows us to apply a custom function to each group.

#### Finding the percentage of hourly temperature below 32
We can use the apply method to help us find the percentage of hourly temps that are below 32 in each month. First we need to define the function we will use.

In [31]:
def prctTempsBelow32(x):
    # x is a series of hourly Air Temp values. All we need to do is test if
    # each value is less than 32. This will create a series of boolean values,
    # a value will be True if the temp is less than 32, and False if not.
    # Finding the mean of this boolean series gives us the percentage of hours
    # that are less than 32 (because True has a value of 1, and False a value
    # of 0)

    # The below code is great, but we could also do it all on one line:
    # return (x < 32).mean()

    tempLessThan32 = (x < 32).mean()
    return tempLessThan32

In [32]:
monthGroups['Air Temp'].apply(prctTempsBelow32) * 100

,Air Temp
Date_Time_Combined,
1,0.0


#### What if we want the odds of any day in the month having at least one or more hourly temps below 32?

In [33]:
def prctDayBelow32(x):
    # x is now a dataframe with rows only corresponding to a specific month
    # we will group this data by the day of the month
    dayGroups = x.groupby(by='Day')
    # now, we just need to test if the min temp on each day is below 32. If
    # this is true, then we know that at least one hours is below 32
    daysLessThan32 = (dayGroups['Air Temp'].min() < 32)
    # daysLessThan32 is no a boolean series that is True if the days min temp
    # is less than 32, and false if not. Findinging the mean of this gives us
    # the percentages of days less than 32

    # This code works, but we could also do it all on one line:
    # return (x.groupby(by='Day')['Air Temp'].min() < 32).mean()
    return daysLessThan32.mean()

In [34]:
monthGroups.apply(prctDayBelow32) * 100

,0
Date_Time_Combined,
1,0.0


#### Quick Review of Lambda Functions
A lambda function is a function that we define _in line_ that does not have a name. (In other programming languages these are sometimes called anonymous functions). For example,
instead of using the prctTempBelow32 function above we could just pass a lambda function to the apply method:

In [35]:
monthGroups['Air Temp'].apply(lambda x: (x < 32).mean())

,Air Temp
Date_Time_Combined,
1,0.0


# Repeating The Same Analysis Using a Datetime Column

#### You can do the same on a datetime column, you just need to pass the datetime column itself to the groupby method:

First I am going to reload the data using the parse_dates argument to convert the first 4 columns to a datetime, and convert the 'Air Temp' column to Fahrenheit.

In [36]:
weatherData = pd.read_csv(filepath, delim_whitespace=True,
                          names=headers, parse_dates=[[0, 1, 2, 3]])
weatherData.loc[:, 'Air Temp'] = (weatherData['Air Temp']/10) * 1.8 + 32
weatherData.head(2)

/tmp/ipykernel_532/3324488776.py:1: FutureWarning: Support for nested sequences for 'parse_dates' in pd.read_csv is deprecated. Combine the desired columns with pd.to_datetime after parsing instead.
  weatherData = pd.read_csv(filepath, delim_whitespace=True,
/tmp/ipykernel_532/3324488776.py:1: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  weatherData = pd.read_csv(filepath, delim_whitespace=True,
/tmp/ipykernel_532/3324488776.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  weatherData = pd.read_csv(filepath, delim_whitespace=True,
/tmp/ipykernel_532/3324488776.py:3: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[46.4 47.3 48.2 49.1 50.  50.9 51.8 52.7 53.6 54.5 55.4 56.3 57.2 

,Year_Month_Day_Hour,Air Temp,Dew Point Temp,Sea Level Pressure,Wind Direction,Wind Speed Rate,Sky Condition Total Coverage Code,Liquid Precipitation Depth Dimension - 1Hr Duration,Liquid Precipitation Depth Dimension - Six Hour Duration
0,2022 1 1 0,46.4,80,10120,180,35,1,0,0
1,2022 1 1 1,47.3,80,10120,180,35,1,0,0


#### Below, note how I pass the datetime column, using .dt.month to get the month of each value in the column
So, really, I am passing a series of month values

In [39]:
weatherData['Year_Month_Day_Hour'] = pd.to_datetime(weatherData['Year_Month_Day_Hour'], format='%Y %m %d %H')
weatherData.groupby(by=weatherData['Year_Month_Day_Hour'].dt.month)['Air Temp'].mean()

,Air Temp
Year_Month_Day_Hour,
1,56.75


## More Groupby Examples On Our Labor Sheet Data
The labor sheet is a great use case for groupby! Naturally, we may want to compare manager performance. Let's use groupy and some analysis to do!

First, we will load the dataset, just like we have before.

In [40]:
filepath = ('/content/drive/MyDrive/Colab Notebooks/UCI/UCI_427.62_Python_for_Data_Analysis/05_Datasets/M5_LaborSheetData.csv')
laborSheetData = pd.read_csv(filepath, parse_dates=[[2, 3], 13])
laborSheetData.head()

/tmp/ipykernel_532/229845300.py:2: FutureWarning: Support for nested sequences for 'parse_dates' in pd.read_csv is deprecated. Combine the desired columns with pd.to_datetime after parsing instead.
  laborSheetData = pd.read_csv(filepath, parse_dates=[[2, 3], 13])
/tmp/ipykernel_532/229845300.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  laborSheetData = pd.read_csv(filepath, parse_dates=[[2, 3], 13])


,Hour_Day_Part,Store,Business_Date,Transactions,Net_Sales,Labor_Hours,Drive_Thru_Avg_Seconds,Crew_Count,Manager,Weather,Promotion_Flag,Customer_Satisfaction,Timestamp
0,00:00 Breakfast,Training Store,2022-01-03,80,650.0,10.00,120,4,Sample Manager,Rain,True,4.00,2022-01-03 00:00:00
1,01:00 Breakfast,Training Store,2022-01-03,87,733.5,11.25,129,5,Sample Manager,Clear,False,4.15,2022-01-03 01:00:00
2,02:00 Breakfast,Training Store,2022-01-03,94,817.0,12.50,138,6,Sample Manager,Clear,False,4.30,2022-01-03 02:00:00
3,03:00 Breakfast,Training Store,2022-01-03,101,900.5,13.75,147,7,Sample Manager,Clear,False,4.45,2022-01-03 03:00:00
4,04:00 Breakfast,Training Store,2022-01-03,108,984.0,15.00,156,8,Sample Manager,Rain,False,4.60,2022-01-03 04:00:00


#### Let's group the laborsheet data by the managers

In [41]:
managerGroups = laborSheetData.groupby('Manager')
# let's print the size of the groups just to get an idea of what we are working with.
managerGroups.size()

,0
Manager,
Sample Manager,42


#### Let's find the manager with the best mean DT TTL (drive through times)
To do this, we will use .mean() and then we will use .sort_values() to sort from the smallest DT TTL (fastest time) to the longest time.

In [43]:
managerGroups['Drive_Thru_Avg_Seconds'].mean().sort_values()

,Drive_Thru_Avg_Seconds
Manager,
Sample Manager,150.214286


#### This is great, but we question Brittany S's time - I wonder how many hours she has worked?
Let's take into account the sample size by also finding the count of values for each manager. We need to use agg() for this. Aso, since the results have more than one column, we need to pass the name of the column that we want to sort by to the sort_values() method.

In [45]:
managerDTTTL = (managerGroups['Drive_Thru_Avg_Seconds'].agg(['mean', 'count']).
                sort_values('mean').rename(columns={'mean': 'Mean DT TTL',
                                                     'count': 'Sample Size'}))
print(managerDTTTL)

                Mean DT TTL  Sample Size
Manager                                 
Sample Manager   150.214286           42


#### Let's filter out managers with a sample size less than 100.

In [46]:
managerDTTTL.loc[managerDTTTL['Sample Size'] > 100, :].sort_values(by='Mean DT TTL')

,Mean DT TTL,Sample Size
Manager,,


#### Let's investigate the relationship between DT TTLs and the hour of the day.

In [48]:
hourGroups = laborSheetData.groupby(laborSheetData['Timestamp'].dt.hour)

In [50]:
hourGroupDTTTLs = hourGroups['Drive_Thru_Avg_Seconds'].agg(['mean', 'count'])
print(hourGroupDTTTLs)

            mean  count
Timestamp              
0          120.0      2
1          129.0      2
2          138.0      2
3          147.0      2
4          156.0      2
5          165.0      2
6          174.0      2
7          183.0      2
8          120.0      2
9          129.0      2
10         138.0      2
11         147.0      2
12         156.0      2
13         165.0      2
14         174.0      2
15         183.0      2
16         120.0      2
17         129.0      2
18         138.0      1
19         147.0      1
20         156.0      1
21         165.0      1
22         174.0      1
23         183.0      1


#### What if we express manager DT TTLs in terms of deviations from the mean?
To do this we need to subtract the mean DT TTL for each of the recorded DT TTL values.
There are multiple ways to do this - we will show one way below.

#### Introducing the .to_dict() method to extract values from a dataframe to a dictionary.
Dictionaries are fast data structures - that are liberally used in python.

In [51]:
meanHourDTTTLs = hourGroupDTTTLs['mean'].to_dict()
print(meanHourDTTTLs)

{0: 120.0, 1: 129.0, 2: 138.0, 3: 147.0, 4: 156.0, 5: 165.0, 6: 174.0, 7: 183.0, 8: 120.0, 9: 129.0, 10: 138.0, 11: 147.0, 12: 156.0, 13: 165.0, 14: 174.0, 15: 183.0, 16: 120.0, 17: 129.0, 18: 138.0, 19: 147.0, 20: 156.0, 21: 165.0, 22: 174.0, 23: 183.0}


#### Now we loop through the dictionary and then subtract the mean from the appropriate rows in the dataframe:

In [53]:
for hour in meanHourDTTTLs:
    laborSheetData.loc[laborSheetData['Timestamp'].dt.hour == hour, 'Drive_Thru_Avg_Seconds'] = \
    laborSheetData.loc[laborSheetData['Timestamp'].dt.hour == hour, 'Drive_Thru_Avg_Seconds'] - meanHourDTTTLs[hour]

In [54]:
laborSheetData.head()

,Hour_Day_Part,Store,Business_Date,Transactions,Net_Sales,Labor_Hours,Drive_Thru_Avg_Seconds,Crew_Count,Manager,Weather,Promotion_Flag,Customer_Satisfaction,Timestamp
0,00:00 Breakfast,Training Store,2022-01-03,80,650.0,10.00,0,4,Sample Manager,Rain,True,4.00,2022-01-03 00:00:00
1,01:00 Breakfast,Training Store,2022-01-03,87,733.5,11.25,0,5,Sample Manager,Clear,False,4.15,2022-01-03 01:00:00
2,02:00 Breakfast,Training Store,2022-01-03,94,817.0,12.50,0,6,Sample Manager,Clear,False,4.30,2022-01-03 02:00:00
3,03:00 Breakfast,Training Store,2022-01-03,101,900.5,13.75,0,7,Sample Manager,Clear,False,4.45,2022-01-03 03:00:00
4,04:00 Breakfast,Training Store,2022-01-03,108,984.0,15.00,0,8,Sample Manager,Rain,False,4.60,2022-01-03 04:00:00


#### Now we use recalculate the mean DT TTLs, using the same groupby object!
Since we did not update anything in the managers column, we do not need to create a new groupby object!

In [59]:
(managerGroups['Drive_Thru_Avg_Seconds'].agg(['mean', 'count']).
                sort_values('mean').rename(columns={'mean': 'Mean DT TTL Deviations',
                                                     'count': 'Sample Size'}))

,Mean DT TTL Deviations,Sample Size
Manager,,
Sample Manager,0.0,42


#### Let's now assign the output to a variable (a new dataframe) which we can then filter by the sample size

In [60]:
managerDTTTLDev = (managerGroups['Drive_Thru_Avg_Seconds'].agg(['mean', 'count']).
                sort_values('mean').rename(columns={'mean': 'Mean DT TTL Deviations',
                                                     'count': 'Sample Size'}))

In [61]:
managerDTTTLDev.loc[managerDTTTLDev.loc[:, 'Sample Size'] > 5, :].sort_values(by='Mean DT TTL Deviations')

,Mean DT TTL Deviations,Sample Size
Manager,,
Sample Manager,0.0,42


![](Success!.png)